In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import re
import time

# Seasons 1, 2, and 3 cover everything up to the end of the Frieza Saga
TARGET_SEASONS = [1, 2, 3]
dataset = []

print("Initializing Dragon Ball Z Wikipedia Scraper...")

for season in TARGET_SEASONS:
    print(f"Fetching Season {season} data...")
    url = f"https://en.wikipedia.org/wiki/Dragon_Ball_Z_season_{season}"

    # Identify yourself politely to Wikipedia's servers
    headers = {
        'User-Agent': 'DBZModelDatasetBuilder/1.0 (contact: your_email@example.com)'
    }

    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"Failed to fetch Season {season}. Status code: {response.status_code}")
        continue

    soup = BeautifulSoup(response.text, 'html.parser')

    # Find all episode tables
    tables = soup.find_all('table', class_='wikitable')

    for table in tables:
        rows = table.find_all('tr')

        for i in range(1, len(rows)):
            row = rows[i]

            # Wikipedia puts the narrative text in a row with class 'description'
            summary_td = row.find('td', class_='description')

            if summary_td:
                plot_text = summary_td.get_text(strip=True)

                # Clean up Wikipedia citation markers like [1], [2], [citation needed]
                plot_text = re.sub(r'\[\d+\]', '', plot_text)
                plot_text = re.sub(r'\[citation needed\]', '', plot_text)

                # The episode meta-data lives in the row immediately above the description
                meta_row = rows[i-1]
                cells = meta_row.find_all(['th', 'td'])

                if len(cells) >= 2:
                    ep_num = cells[0].get_text(strip=True)
                    # Clean up quotes around titles
                    title = cells[1].get_text(strip=True).replace('"', '')

                    dataset.append({
                        "season": season,
                        "episode_number": ep_num,
                        "title": title,
                        "plot_summary": plot_text
                    })

    # Sleep to avoid hitting Wikipedia's servers too hard
    time.sleep(2)

# Save your structured data to a JSON file
output_file = 'dbz_frieza_saga_dataset.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(dataset, f, indent=4, ensure_ascii=False)

print(f"\nSuccess! Saved {len(dataset)} episodes to {output_file}.")

In [ ]:
import requests
import json
import time

# The exact Wikipedia article names for your core seed characters/lore
TARGET_PAGES = {
    "Goku": "Goku",
    "Vegeta": "Vegeta",
    "Frieza": "Frieza",
    "Saiyans": "Saiyan (Dragon Ball)"
}

knowledge_tree_seed = {}

print("🌌 Initializing Capsule Corp Corderz Core Lore Scraper...")

for character, page_title in TARGET_PAGES.items():
    print(f"Retrieving canon matrix data for: {character}...")

    # Official MediaWiki API endpoint to pull clean, unformatted plain-text introductions
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "prop": "extracts",
        "exintro": True,       # Pull only the core introductory summary paragraph
        "explaintext": True,   # Strip out HTML tags, keep it pure raw text
        "titles": page_title,
        "format": "json"
    }

    headers = {
        'User-Agent': 'CapsuleCorpCorderzBot/1.0 (contact: yourteam@example.com)'
    }

    response = requests.get(url, params=params, headers=headers)

    if response.status_code == 200:
        data = response.json()
        page_data = list(data["query"]["pages"].values())[0]

        if "extracts" in page_data:
            text_summary = page_data["extracts"]

            # Map out your RAG structured metadata object
            knowledge_tree_seed[character] = {
                "character_id": character.lower(),
                "canonical_name": character,
                "source": f"https://en.wikipedia.org/wiki/{page_title}",
                "canon_lore_summary": text_summary,
                "alt_timeline_notes": "" # Left blank intentionally for your team to write custom prompt injects!
            }
        else:
            print(f"⚠️ Could not extract text for {character}.")
    else:
        print(f"❌ API Request failed for {character} with code {response.status_code}")

    time.sleep(1) # Polite cooldown delay

# Adding specialized manually-seeded baseline entries for Raditz and Nappa
# (Since they share a general wiki list page, targeted summaries work best for RAG context)
knowledge_tree_seed["Raditz"] = {
    "character_id": "raditz",
    "canonical_name": "Raditz",
    "source": "Manual Seed",
    "canon_lore_summary": "Goku's biological older brother and a mid-to-low class Saiyan warrior. Arrives on Earth to recruit Kakarot into the Frieza Force.",
    "alt_timeline_notes": "In this timeline, Raditz arrives on Earth to find Kakarot has successfully wiped out humanity. Instead of fighting, they immediately unite."
}

knowledge_tree_seed["Nappa"] = {
    "character_id": "nappa",
    "canonical_name": "Nappa",
    "source": "Manual Seed",
    "canon_lore_summary": "An elite Saiyan warrior and general who serves as Vegeta's primary cohort. Renowned for raw physical power and a destructive temper.",
    "alt_timeline_notes": "Views Kakarot as a weakling low-class warrior because Kakarot lacks the refined combat multipliers developed through Earth's unique training systems."
}

# Save out the structural document for Milestone 3
output_filename = "capsule_corp_knowledge_seed.json"
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(knowledge_tree_seed, f, indent=4, ensure_ascii=False)

print(f"\n✅ Mission Accomplished! Core dataset written to: {output_filename}")

In [ ]:
import json

# Load the timeline episodes
with open('dbz_frieza_saga_dataset.json', 'r', encoding='utf-8') as f:
    timeline_data = json.load(f)

# Load the character profiles
with open('capsule_corp_knowledge_seed.json', 'r', encoding='utf-8') as f:
    character_data = json.load(f)

# Combine them into a single master dictionary
master_dataset = {
    "character_profiles": character_data,
    "canonical_timeline": timeline_data
}

# Save the unified master file
with open('dbz_master_dataset.json', 'w', encoding='utf-8') as f:
    json.dump(master_dataset, f, indent=4, ensure_ascii=False)

print("⚡ Master dataset unified successfully as 'dbz_master_dataset.json'!")

In [ ]:
!pip install -q sentence-transformers numpy

In [ ]:
from sentence_transformers import SentenceTransformer
import json
import numpy as np

# 1. Load the open-source embedding model (totally free, runs inside Colab)
print("📥 Loading free local embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Load your master data
with open('dbz_master_dataset.json', 'r', encoding='utf-8') as f:
    master_data = json.load(f)

documents = []
print("🚀 Extracting text blocks...")

# Clean data extraction
for char_name, details in master_data["character_profiles"].items():
    text_content = f"Character: {details['canonical_name']}. Core Lore: {details['canon_lore_summary']}"
    documents.append({"source": f"Profile: {char_name}", "text": text_content})

for ep in master_data["canonical_timeline"]:
    text_content = f"Season {ep['season']} Episode {ep['episode_number']}: {ep['title']}. Plot: {ep['plot_summary']}"
    documents.append({"source": f"Episode {ep['episode_number']}", "text": text_content})

# 3. Generate Vectors locally
print(f"Converting {len(documents)} chunks into vectors. No API keys required!")
raw_texts = [doc["text"] for doc in documents]
vector_matrix = model.encode(raw_texts, show_progress_bar=True)

# 4. Search Function
def free_vector_search(player_query, top_k=2):
    query_vector = model.encode([player_query])[0]

    # Calculate math matches
    dot_products = np.dot(vector_matrix, query_vector)
    matrix_norms = np.linalg.norm(vector_matrix, axis=1)
    query_norm = np.linalg.norm(query_vector)
    similarities = dot_products / (matrix_norms * query_norm)

    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "source": documents[idx]["source"],
            "text": documents[idx]["text"],
            "score": float(similarities[idx])
        })
    return results

print("✅ Local Vector Engine Ready!")

In [ ]:
# Test your system using the local AI models
test_input = "Vegeta arrives on Earth looking for Kakarot"
matches = free_vector_search(test_input, top_k=1)

print(f"Match found from source: {matches[0]['source']}\n")
print(matches[0]['text'])